# VirulentHunter benchmark

In [ ]:
import pandas as pd

df_train = pd.read_csv('binary/train_labels.csv')
df_val = pd.read_csv('binary/val_labels.csv')
df_test = pd.read_csv('binary/test_labels.csv')

In [ ]:
df_train = pd.concat([df_train, df_val])
df_train

,Unnamed: 0,id,label
0,7469,VFG016628,1
1,7273,VFG031691,1
2,724,VFG000676,1
3,11873,gi|13449090|ref|NP_085306.1|,1
4,5601,VFG045743,1
...,...,...,...
2878,9501,VFG040343,1
2879,2433,sp|P9WPN3|CP132_MYCTU,0
2880,5950,VFG008206,1
2881,728,sp|Q10284|SSB1_SCHPO,0


In [ ]:
import numpy as np
import os
import torch
from tqdm import tqdm

def load_multiple_embeddings(df, emb_dirs):
    """
    Load and concatenate embeddings from multiple directories for the same ids in df.

    Args:
        df: DataFrame with columns ['id', 'label']
        emb_dirs: list of 3 paths to embedding directories

    Returns:
        features: np.ndarray of concatenated embeddings from all three sources
        labels: np.ndarray of labels
    """
    features = []
    labels = []

    for idx in tqdm(range(len(df)), desc="Loading embeddings"):
        id_ = df.iloc[idx]['id']
        label = df.iloc[idx]['label']

        emb_list = []
        for emb_dir in emb_dirs:
            emb_path = os.path.join(emb_dir, f'{id_}.pt')
            emb = torch.load(emb_path)
            emb_np = emb.numpy()

            if emb_np.ndim == 2:
                emb_np = emb_np.mean(axis=0)
            elif emb_np.ndim != 1:
                raise ValueError(f'Unexpected embedding shape: {emb_np.shape} for id {id_} in {emb_dir}')

            emb_list.append(emb_np)

        concatenated_emb = np.concatenate(emb_list)
        features.append(concatenated_emb)
        labels.append(label)

    features = np.stack(features)
    labels = np.array(labels)
    return features, labels


/home/sjchoi/anaconda3/envs/node08/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# reduced duplicates

In [ ]:
esm_dir = '../VirulentHunter/esm_emb'
interproscan_dir = '../VirulentHunter/Bert_emb_interproscan_nodup_semantic'
taxonomy_dir ='../VirulentHunter/Bert_emb_gtdb'

X_train, y_train = load_multiple_embeddings(df_train, [esm_dir, interproscan_dir, taxonomy_dir])
X_test, y_test = load_multiple_embeddings(df_test, [esm_dir, interproscan_dir, taxonomy_dir])

Loading embeddings: 100%|██████████| 2884/2884 [02:42<00:00, 17.76it/s]


In [ ]:
import warnings
warnings.filterwarnings(action='ignore', category=FutureWarning, module='xgboost')

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef

# Define the parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0]
}

# Initialize XGBClassifier
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Grid search with 5-fold cross-validation
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=5, scoring='roc_auc', verbose=1, n_jobs=-1)

# Fit grid search
grid_search.fit(X_train, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation AUC: ", grid_search.best_score_)

# Evaluate on test data using best estimator
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]


In [ ]:
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")

Test ROC AUC: 0.9649457718802479
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.91      0.91      1442
           1       0.91      0.90      0.91      1442

    accuracy                           0.91      2884
   macro avg       0.91      0.91      0.91      2884
weighted avg       0.91      0.91      0.91      2884

Accuracy: 0.907
F1-score: 0.907
Precision: 0.913
Recall: 0.901
MCC: 0.815


In [ ]:
import joblib

# After training and grid search as before
best_model = grid_search.best_estimator_

# Save the model to a file
joblib.dump(best_model, 'best_binary_xgb_model.pkl')

# Later, you can load it back with
# loaded_model = joblib.load('best_xgb_model.pkl')


['best_binary_xgb_model.pkl']

In [ ]:
import pandas as pd
import joblib

# Load the trained model
loaded_model = joblib.load('best_binary_xgb_model.pkl')

# Load your test set (replace 'test.csv' with your actual test file path)
df_test = pd.read_csv('binary/test_labels.csv')

labels = df_test['label']

# Predict
probabilities = loaded_model.predict_proba(X_test)[:,1]
predictions = loaded_model.predict(X_test)


# Prepare result dataframe with id, predicted label, and original label
result_df = pd.DataFrame({
    'id': df_test['id'],
    'prob': probabilities,
    'pred': predictions,
    'label': labels
})

# Save results
result_df.to_csv('binary/VFTextSeq_prediction_best_xgb.csv', index=False)


In [ ]:
result_df